# 05 Design spike

Phase 5 starts with a one-day spike. Each cell tests one idea; record pass or fail in the table at the end and screenshot anything odd.

Needs internet: tiles come from OpenFreeMap and the AWS elevation tiles, both keyless.

In [14]:
import warnings
import webbrowser
from pathlib import Path

import geopandas as gpd
import ipywidgets as widgets
from IPython.display import display
from lonboard import Map
from lonboard.basemap import MaplibreBasemap

import kharita as kh
from kharita import basemaps as kb

warnings.filterwarnings("ignore", module="lonboard")
states = gpd.read_file("../data/processed/sudan_states.gpkg")
points = states.assign(geometry=states.representative_point())
edges = states.assign(geometry=states.boundary)
SUDAN = {"longitude": 30.0, "latitude": 15.5, "zoom": 4.3}
CHORO = "fill-[density] ramp-ylorrd scheme-q5 stroke-1 stroke-white/60 opacity-85"

In [15]:
# A. Themed basemap from a base64 data URL (paper)
Map(kh.layer(states, CHORO), basemap=kb.basemap("paper"), view_state=SUDAN, height=420)

In [16]:
# B. Same theme, percent-encoded data URL
Map(kh.layer(states, CHORO), basemap=kb.basemap("paper", encoding="percent"), view_state=SUDAN, height=420)

In [17]:
# C. Same theme passed as a style object, no URL at all
Map(kh.layer(states, CHORO), basemap=kb.basemap("paper", encoding="object"), view_state=SUDAN, height=420)

In [18]:
# D. Every theme: muted has no labels, midnight and neon are dark
for name in kb.THEMES:
    display(widgets.HTML(f"<b>{name}</b>"))
    display(Map(kh.layer(states, "fill-transparent stroke-1 stroke-rose-500"), basemap=kb.basemap(name), view_state=SUDAN, height=300))

HTML(value='<b>paper</b>')

HTML(value='<b>midnight</b>')

HTML(value='<b>mono</b>')

HTML(value='<b>neon</b>')

HTML(value='<b>muted</b>')

In [19]:
# E. Hillshade from keyless elevation tiles: Jebel Marra, Darfur, then the whole country
JEBEL_MARRA = {"longitude": 24.3, "latitude": 13.0, "zoom": 7}
outline = "fill-transparent stroke-1 stroke-slate-700"
display(Map(kh.layer(states, outline), basemap=kb.basemap("paper", hillshade=True), view_state=JEBEL_MARRA, height=420))
display(Map(kh.layer(states, outline), basemap=kb.basemap("midnight", hillshade=True), view_state=SUDAN, height=420))

In [20]:
# F. Labels above the data: interleaved mode, kharita layers drawn below the first label layer
layers = kh.layer(states, CHORO)
for lyr in layers:
    lyr.before_id = kb.FIRST_LABEL
Map(layers, basemap=kb.basemap("paper", mode="interleaved"), view_state=SUDAN, height=420)

In [21]:
# G. Glow: additive blending through Map.parameters, luma.gl v9 names; top map is the control
GLOW = {
    "blend": True,
    "blendColorOperation": "add",
    "blendColorSrcFactor": "src-alpha",
    "blendColorDstFactor": "one",
    "blendAlphaOperation": "add",
    "blendAlphaSrcFactor": "one",
    "blendAlphaDstFactor": "one-minus-src-alpha",
    "depthCompare": "always",
}


def glow_layers():
    # A wide faint disc under a small bright core
    return kh.layer(points, "fill-[pop] ramp-plasma size-28 opacity-35") + kh.layer(points, "fill-[pop] ramp-plasma size-6")


display(widgets.HTML("<b>Without glow</b>"))
display(Map(glow_layers(), basemap=kb.basemap("neon"), view_state=SUDAN, height=360))
display(widgets.HTML("<b>With glow</b>"))
display(Map(glow_layers(), basemap=kb.basemap("neon"), parameters=GLOW, view_state=SUDAN, height=360))

HTML(value='<b>Without glow</b>')

HTML(value='<b>With glow</b>')

In [22]:
# H. Glow with legacy WebGL constants, only matters if G shows no difference
LEGACY = {"blendFunc": [770, 1, 1, 771], "blendEquation": 32774, "depthTest": False}
Map(glow_layers(), basemap=kb.basemap("neon"), parameters=LEGACY, view_state=SUDAN, height=360)

In [23]:
# I. Halo: two wide faint outlines under the data, on midnight
halo = (
    kh.layer(edges, "stroke-10 stroke-cyan-300/15")
    + kh.layer(edges, "stroke-4 stroke-cyan-300/30")
    + kh.layer(states, "fill-[density] ramp-viridis scheme-q5 stroke-1 stroke-cyan-200 opacity-90")
)
Map(halo, basemap=kb.basemap("midnight"), view_state=SUDAN, height=420)

In [24]:
# J. Keyless hosted dark styles that could replace CARTO Dark Matter
for url in ("https://tiles.openfreemap.org/styles/dark", "https://tiles.openfreemap.org/styles/fiord"):
    display(widgets.HTML(f"<b>{url}</b>"))
    display(Map(kh.layer(states, CHORO), basemap=MaplibreBasemap(style=url), view_state=SUDAN, height=300))

HTML(value='<b>https://tiles.openfreemap.org/styles/dark</b>')

HTML(value='<b>https://tiles.openfreemap.org/styles/fiord</b>')

In [25]:
# K. Export: the theme style travels inside the HTML file
out = Path("../spike_out")
out.mkdir(exist_ok=True)
exported = Map(kh.layer(states, CHORO), basemap=kb.basemap("midnight", hillshade=True), view_state=SUDAN)
(out / "theme_export.html").write_text(exported.to_html(title="Theme export"), encoding="utf-8")
webbrowser.open((out / "theme_export.html").resolve().as_uri())

True

In [26]:
# L. Legend fix: the bottom row now clears the square button in the corner
kh.map(states, "preset-choropleth fill-[density] ramp-ylorrd", legend={"title": "People per km2"})

## Results

Fill in and send back with screenshots of any failures.

| Test | What passing looks like | Pass? | Notes |
| --- | --- | --- | --- |
| A | Cream ground, blue-grey water, place labels | yes | works properly |
| B | Same as A | yes | not sure what is the point of htis test |
| C | Same as A | yes | also not sure what is this test for |
| D | Five distinct themes; muted without labels | yes | created 5 maps without lables just design of the map of paper midnight mono neon muted |
| E | Visible relief around Jebel Marra; subtle at country zoom | yes but some issue i think | so the first map is working for the alivation but the second one im not sure what is it for |
| F | City and state names drawn on top of the colours | yes | works good |
| G | Bottom map brighter where discs overlap | yes | it words for both with and without glow |
| H | Same as G bottom, or an error | yes | g worked for no need for this |
| I | Soft cyan glow along borders | yes | i can see the halo |
| J | Both dark styles load | yes | one is blue backgound and one is black but we cna use both and include then to be for different uses |
| K | Exported page shows the midnight theme | yes | the dark theme website works |
| L | Legend's last row fully visible | yes | the legend have been moved above the small square so everything in it is now visable |